In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from pathlib import Path

from src.data.enron import build_enron
from src.data.splits import make_splits
from src.features.extractor import extract_frame, FEATURES
from src.profiling.stats import icc1

ENRON_ROOT = "../data/external/Enron/maildir"
MIN_MSGS = 100

raw = build_enron(ENRON_ROOT)
df = make_splits(raw, min_msgs=MIN_MSGS)
df = extract_frame(df)
df.to_csv("../data/processed/enron_messages.csv", index=False)
print(f"{len(raw):,} cleaned sent messages -> {len(df):,} from {df.user_id.nunique()} users (>= {MIN_MSGS} msgs)")

per_user = df.groupby("user_id").agg(n=("message", "size"),
                                     first=("timestamp", "min"), last=("timestamp", "max"),
                                     n_recipients_seen=("recipient_id", "nunique"))
print(per_user["n"].describe().round(1))
print("share with quoted context:", (df["context"].str.len() > 0).mean().round(3))
print(df["user_split"].value_counts().to_dict(), df["time_split"].value_counts().to_dict())
per_user.head()

nz = (df[FEATURES] != 0).mean().round(3).rename("nonzero_rate")
nz

KEY = ["log_words", "avg_sent_len", "punct_density", "n_questions", "n_exclam",
       "caps_ratio", "starts_lower", "has_newline", "has_list"]
pair = df[df.recipient_id != ""].copy()
pair["pair"] = pair["user_id"] + "_" + pair["recipient_id"]
pair = pair[pair.groupby("pair")["pair"].transform("size") >= 3]
res = pd.DataFrame({
    "user_ICC": {f: icc1(df[f], df["user_id"]) for f in KEY},
    "user_x_recipient_ICC": {f: icc1(pair[f], pair["pair"]) for f in KEY},
}).round(3)
res

has_ctx = df[df["context"].str.len() > 0].copy()
has_ctx["ctx_log_words"] = np.log1p(has_ctx["context"].str.split().str.len())
print("spearman(reply_len, context_len):",
      round(has_ctx[["log_words", "ctx_log_words"]].corr(method="spearman").iloc[0, 1], 3))


KeyError: 'user_id'